# <font color="#418FDE" size="6.5" uppercase>**Workflow and Baselines**</font>

>Last update: 20260709.
    
By the end of this Lecture, you will be able to:
- Describe the steps in an end-to-end machine learning workflow for civil engineering data. 
- Implement a manual train/test split using simple Python and NumPy operations. 
- Build baseline predictors for regression and classification before training complex models. 


## **1. ML Pipeline**

### **1.1. Problem Framing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_01_01.jpg?v=1783612915" width="250">



>* Turn engineering needs into precise ML tasks
>* Define success before building the model

>* Define inputs, outputs, scope, and timing
>* Account for constraints shaping later workflow

>* Judge models by real engineering consequences
>* Compare results against practical baseline methods



In [ ]:
#@title Python Code - Problem Framing

# Frame engineering needs before modeling.
# Define inputs outputs and success.
# Compare tasks before choosing methods.

import pandas as pd
import numpy as np

# Create small civil engineering examples.
projects = pd.DataFrame({
    "need": ["reduce bridge congestion", "prioritize pavement repairs", "screen slope failures"],
    "input": ["hour, weather, lane closures", "age, traffic, cracks", "rainfall, soil, slope angle"],
    "output": ["hourly traffic volume", "condition rating", "failure risk class"],
    "type": ["regression", "regression", "classification"],
    "baseline": ["historical hourly average", "last inspection rating", "majority risk class"]
})

# Check the table is usable.
assert projects.shape == (3, 5)
assert projects["output"].notna().all()

# Score framing completeness simply.
required_columns = ["need", "input", "output", "type", "baseline"]
projects["framing_score"] = projects[required_columns].notna().sum(axis=1)

# Select one framed problem.
chosen = projects.loc[1]
print("Problem framing turns needs into prediction tasks.")
print("Chosen need:", chosen["need"])
print("Inputs available:", chosen["input"])
print("Target output:", chosen["output"])
print("Task type:", chosen["type"])
print("Baseline comparison:", chosen["baseline"])
print("Framing score:", int(chosen["framing_score"]), "of", len(required_columns))



### **1.2. Data Preparation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_01_02.jpg?v=1783612913" width="250">



>* Gather and inspect diverse infrastructure data
>* Use engineering judgment to ensure reliability

>* Clean missing values and investigate outliers
>* Standardize units, timestamps, labels, and records

>* Create task-specific features and targets
>* Prevent leakage for fair model evaluation



In [ ]:
#@title Python Code - Data Preparation

# Prepare civil engineering data for modeling.
# Clean units, missing values, and labels.
# Create features and targets carefully.

# Import small, standard teaching libraries.
import numpy as np
import pandas as pd

# Create a tiny bridge inspection dataset.
data = pd.DataFrame({
    "bridge_id": [101, 102, 103, 104, 105, 106],
    "span_ft": [120, 85, 200, 150, np.nan, 95],
    "age_years": [12, 35, 8, 50, 22, 40],

    "traffic_vpd": [18000, 9000, 25000, 12000, 15000, 8000],
    "deck_rating": [7, 5, 8, 4, 6, np.nan],
    "material": ["Steel", "concrete", "Steel", "Concrete", "steel", "Concrete"]
})

# Standardize categorical labels for consistency.
data["material"] = data["material"].str.lower()

# Convert span from feet to meters.
data["span_m"] = data["span_ft"] * 0.3048

# Fill missing numeric values using medians.
for column in ["span_m", "deck_rating"]:
    data[column] = data[column].fillna(data[column].median())

# Create a simple engineered feature.
data["traffic_per_meter"] = data["traffic_vpd"] / data["span_m"]

# Define features and target for modeling.
features = data[["span_m", "age_years", "traffic_per_meter"]]
target = data["deck_rating"]

# Validate prepared shapes before modeling.
assert features.shape[0] == target.shape[0]
assert features.isna().sum().sum() == 0

# Print a compact preparation summary.
print("Prepared rows:", len(data))
print("Feature columns:", list(features.columns))
print("Target column: deck_rating")
print("Missing feature values:", int(features.isna().sum().sum()))
print("First prepared bridge:", features.iloc[0].round(2).to_dict())



### **1.3. Train Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_01_03.jpg?v=1783612917" width="250">



>* Train models to learn engineering data patterns
>* Match algorithms to data, errors, and consequences

>* Start with baselines before complex models
>* Require clear improvement for engineering decisions

>* Test models on unseen data
>* Balance accuracy with practical engineering needs



In [ ]:
#@title Python Code - Train Models

# Train simple models after preparing civil data.
# Baselines show whether learning adds value.
# Manual splitting supports fair model evaluation.

# Import NumPy for small numerical arrays.
import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(42)

# Create synthetic concrete mixture data.
cement = np.array([280, 300, 320, 340, 360, 380, 400, 420], dtype=float)
water = np.array([190, 185, 180, 175, 170, 165, 160, 155], dtype=float)

# Define compressive strength in megapascals.
strength = np.array([28, 31, 34, 37, 40, 43, 46, 49], dtype=float)

# Stack features into one design matrix.
X = np.column_stack([cement, water])
y = strength.copy()

# Validate matching sample counts before splitting.
assert X.shape[0] == y.shape[0]

# Shuffle indices for a manual train test split.
indices = rng.permutation(len(y))
train_size = int(0.75 * len(y))

# Select training and testing row indices.
train_idx = indices[:train_size]
test_idx = indices[train_size:]

# Build train and test arrays manually.
X_train = X[train_idx]
y_train = y[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]

# Baseline predicts the training mean strength.
baseline_pred = np.full_like(y_test, y_train.mean())

# Add an intercept column for linear regression.
ones_train = np.ones((X_train.shape[0], 1))
X_train_aug = np.column_stack([ones_train, X_train])

# Fit coefficients using least squares.
coefficients = np.linalg.lstsq(X_train_aug, y_train, rcond=None)[0]

# Predict test strengths with the trained model.
ones_test = np.ones((X_test.shape[0], 1))
X_test_aug = np.column_stack([ones_test, X_test])
model_pred = X_test_aug @ coefficients

# Compute mean absolute error for comparison.
baseline_mae = np.mean(np.abs(y_test - baseline_pred))
model_mae = np.mean(np.abs(y_test - model_pred))

# Print a compact workflow summary.
print("Manual split: train samples =", len(y_train), "test samples =", len(y_test))
print("Baseline MAE, MPa:", round(float(baseline_mae), 2))
print("Trained linear model MAE, MPa:", round(float(model_mae), 2))
print("Model coefficients:", np.round(coefficients, 3).tolist())
print("Training lesson: compare simple baselines before trusting complex models.")



## **2. Manual Data Splitting**

### **2.1. Random Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_02_01.jpg?v=1783612920" width="250">



>* Shuffle data into training and test sets
>* Evaluate models on unseen examples

>* Shuffle row indices, not data meanings
>* Keep features and labels aligned

>* Use random splits for independent, unordered data
>* Avoid leakage from time or grouped records



In [ ]:
#@title Python Code - Random Split

# Manual random splitting keeps records aligned.
# NumPy indices make shuffling transparent.
# Civil datasets need careful evaluation.

# Import NumPy for arrays and random shuffling.
import numpy as np

# Create a deterministic random number generator.
rng = np.random.default_rng(seed=42)

# Store bridge features as aligned columns.
features = np.array([
    [12, 80, 18],
    [35, 120, 32],
    [7, 60, 12],
    [50, 200, 45],

    [22, 95, 25],
    [15, 110, 20],
    [40, 150, 38],
    [5, 55, 10],
])

# Store condition ratings for matching rows.
target = np.array([82, 65, 90, 48, 74, 79, 58, 93])

# Validate that features and target align.
assert features.shape[0] == target.shape[0]

# Count rows and choose training fraction.
n_rows = features.shape[0]
train_fraction = 0.75

# Convert the fraction into row counts.
n_train = int(n_rows * train_fraction)
assert 0 < n_train < n_rows

# Shuffle row positions, not column values.
all_indices = np.arange(n_rows)
shuffled_indices = rng.permutation(all_indices)

# Slice shuffled indices into two groups.
train_indices = shuffled_indices[:n_train]
test_indices = shuffled_indices[n_train:]

# Use indices to keep rows aligned.
X_train = features[train_indices]
y_train = target[train_indices]

# Create the held-out test arrays.
X_test = features[test_indices]
y_test = target[test_indices]

# Check that split sizes are sensible.
assert X_train.shape[0] + X_test.shape[0] == n_rows

# Print a compact summary for learners.
print("Total records:", n_rows)
print("Training records:", X_train.shape[0])
print("Test records:", X_test.shape[0])
print("Shuffled row order:", shuffled_indices.tolist())
print("First training row features:", X_train[0].tolist())
print("Matching training target:", int(y_train[0]))



### **2.2. Train Test Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_02_02.jpg?v=1783612919" width="250">



>* Train on one part, test on another
>* Separate data for fair future performance estimates

>* Choose split size and shuffle row indices
>* Keep features and targets correctly aligned

>* Respect data groups and time order
>* Match splitting strategy to real use



In [ ]:
#@title Python Code - Train Test Split

# Manual splitting supports honest model evaluation.
# Civil engineering rows need aligned targets.
# NumPy indices make splitting transparent.

# Import NumPy for small array operations.
import numpy as np

# Create reproducible random generator.
rng = np.random.default_rng(42)

# Store six retaining wall observations.
wall_data = np.array([
    [12.0, 0.8, 3.0],
    [18.0, 1.1, 4.5],

    [25.0, 1.6, 6.0],
    [30.0, 2.0, 7.5],
    [35.0, 2.4, 9.0],
    [40.0, 2.9, 10.5],
])

# Store measured wall movement targets.
movement_mm = np.array([2.1, 2.8, 4.0, 5.2, 6.1, 7.4])

# Validate matching observation counts.
assert wall_data.shape[0] == movement_mm.shape[0]

# Count rows and choose training fraction.
n_rows = wall_data.shape[0]
train_fraction = 0.67

# Shuffle row indices reproducibly.
indices = rng.permutation(n_rows)
train_size = int(round(train_fraction * n_rows))

# Split indices into training and testing.
train_idx = indices[:train_size]
test_idx = indices[train_size:]

# Select matching feature and target rows.
X_train = wall_data[train_idx]
y_train = movement_mm[train_idx]
X_test = wall_data[test_idx]
y_test = movement_mm[test_idx]

# Validate split sizes and alignment.
assert len(X_train) + len(X_test) == n_rows
assert len(y_train) == len(X_train)
assert len(y_test) == len(X_test)

# Print compact teaching results.
print("Manual train/test split using NumPy indices")
print(f"Total rows: {n_rows}")
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print(f"Train indices: {train_idx.tolist()}")
print(f"Test indices: {test_idx.tolist()}")
print(f"First training target: {y_train[0]} mm")



### **2.3. Train Test Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_02_03.jpg?v=1783612923" width="250">



>* Train on one part, test on another
>* Separate data gives fairer future performance estimates

>* Shuffle row indices for reproducible splitting
>* Assign proportions to train and test sets

>* Respect data structure when splitting records
>* Prevent leakage with realistic test sets



In [ ]:
#@title Python Code - Train Test Split

# Manual splitting supports honest model evaluation.
# Civil datasets need reproducible row separation.
# NumPy indices make splitting transparent.

# Import NumPy for arrays and shuffling.
import numpy as np

# Create a small concrete strength dataset.
cement_kg = np.array([300, 320, 280, 350, 310, 330, 290, 360])

# Store water cement ratios separately.
water_ratio = np.array([0.55, 0.50, 0.60, 0.45, 0.52, 0.48, 0.58, 0.44])

# Combine features into one matrix.
features = np.column_stack((cement_kg, water_ratio))

# Store target strengths in megapascals.
strength_mpa = np.array([32, 38, 28, 45, 35, 41, 30, 48])

# Validate matching row counts before splitting.
assert features.shape[0] == strength_mpa.shape[0]

# Count observations and choose split size.
n_rows = features.shape[0]
test_fraction = 0.25

# Create reproducible shuffled row indices.
rng = np.random.default_rng(seed=42)
indices = np.arange(n_rows)

# Shuffle indices without changing original arrays.
rng.shuffle(indices)
test_size = int(round(n_rows * test_fraction))

# Separate shuffled indices into train and test.
test_indices = indices[:test_size]
train_indices = indices[test_size:]

# Use indices to create split arrays.
X_train = features[train_indices]
y_train = strength_mpa[train_indices]

# Create held back test arrays.
X_test = features[test_indices]
y_test = strength_mpa[test_indices]

# Validate that every row appears once.
assert len(train_indices) + len(test_indices) == n_rows
assert set(train_indices).isdisjoint(set(test_indices))

# Print compact results for learners.
print("Total rows:", n_rows)
print("Train rows:", len(train_indices))
print("Test rows:", len(test_indices))
print("Train indices:", train_indices.tolist())
print("Test indices:", test_indices.tolist())
print("First train feature row:", X_train[0].tolist())
print("First train target MPa:", int(y_train[0]))



## **3. Baseline Models**

### **3.1. Regression Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_03_01.jpg?v=1783612906" width="250">



>* Simple predictions set a comparison standard
>* Complex models should beat the baseline

>* Use mean, median, or grouped averages
>* Keep baselines simple, fair, and reproducible

>* Compare model errors against simple baselines
>* Use baselines to reveal workflow problems



In [ ]:
#@title Python Code - Regression Baselines

# Regression baselines compare simple prediction rules.
# Civil engineering targets are often continuous.
# We evaluate baselines before complex models.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for reproducible splitting.
rng = np.random.default_rng(42)

# Create small concrete strength example data.
cement_kg = np.array([260, 280, 300, 320, 340, 360, 380, 400])
strength_mpa = np.array([24, 27, 29, 31, 35, 37, 39, 43])

# Validate matching feature and target lengths.
assert cement_kg.shape[0] == strength_mpa.shape[0]

# Shuffle indices before manual train test split.
indices = rng.permutation(strength_mpa.size)
split_point = int(0.75 * strength_mpa.size)

# Select training and testing indices manually.
train_idx = indices[:split_point]
test_idx = indices[split_point:]

# Build train and test target arrays.
y_train = strength_mpa[train_idx]
y_test = strength_mpa[test_idx]

# Compute mean and median regression baselines.
mean_value = np.mean(y_train)
median_value = np.median(y_train)

# Predict the same baseline value for tests.
mean_pred = np.full(y_test.shape, mean_value)
median_pred = np.full(y_test.shape, median_value)

# Define mean absolute error using NumPy.
def mean_absolute_error(actual, predicted):
    return np.mean(np.abs(actual - predicted))

# Evaluate both simple baseline predictors.
mean_mae = mean_absolute_error(y_test, mean_pred)
median_mae = mean_absolute_error(y_test, median_pred)

# Print concise results for comparison.
print("Regression baseline example: concrete strength.")
print(f"Training samples: {y_train.size}, test samples: {y_test.size}.")
print(f"Mean baseline prediction: {mean_value:.1f} MPa.")
print(f"Median baseline prediction: {median_value:.1f} MPa.")
print(f"Mean baseline MAE: {mean_mae:.2f} MPa.")
print(f"Median baseline MAE: {median_mae:.2f} MPa.")

# Plot actual test values and baseline predictions.
plt.figure(figsize=(6, 4))
plt.scatter(test_idx, y_test, label="Actual test strength")
plt.scatter(test_idx, mean_pred, label="Mean baseline")
plt.scatter(test_idx, median_pred, label="Median baseline")
plt.xlabel("Original specimen index")
plt.ylabel("Compressive strength, MPa")
plt.title("Regression Baselines for Concrete Strength")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **3.2. Regression Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_03_02.jpg?v=1783612910" width="250">



>* Simple predictions set a performance benchmark
>* Complex models must beat basic baselines

>* Use mean, median, or domain rules
>* Compare added modeling value against simple benchmarks

>* Compare models on identical held-out test data
>* Use baselines to judge practical model value



In [ ]:
#@title Python Code - Regression Baselines

# Regression baselines compare simple prediction rules.
# Civil engineering targets often contain outliers.
# Test data must stay held out.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for reproducible splitting.
rng = np.random.default_rng(42)

# Create small bridge repair cost data.
ages_years = np.array([4, 7, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36])
repair_costs = np.array([8, 9, 11, 13, 15, 18, 21, 24, 28, 32, 36, 95])

# Validate matching one dimensional arrays.
assert ages_years.ndim == repair_costs.ndim == 1
assert ages_years.size == repair_costs.size

# Shuffle indices before manual splitting.
indices = np.arange(repair_costs.size)
rng.shuffle(indices)

# Use seventy five percent for training.
split_index = int(0.75 * indices.size)
train_idx = indices[:split_index]
test_idx = indices[split_index:]

# Separate targets for baseline summaries.
y_train = repair_costs[train_idx]
y_test = repair_costs[test_idx]

# Build mean and median baseline predictions.
mean_value = np.mean(y_train)
median_value = np.median(y_train)

# Repeat each baseline value for test cases.
mean_pred = np.full(y_test.shape, mean_value)
median_pred = np.full(y_test.shape, median_value)

# Define mean absolute error manually.
def mean_absolute_error(actual, predicted):
    errors = np.abs(actual - predicted)
    return np.mean(errors)

# Evaluate both baselines on identical test data.
mean_mae = mean_absolute_error(y_test, mean_pred)
median_mae = mean_absolute_error(y_test, median_pred)

# Print concise results for comparison.
print(f"Training cases: {y_train.size}, test cases: {y_test.size}")
print(f"Mean baseline prediction: ${mean_value:.1f}k")
print(f"Median baseline prediction: ${median_value:.1f}k")
print(f"Mean baseline MAE: ${mean_mae:.1f}k")
print(f"Median baseline MAE: ${median_mae:.1f}k")

# Plot actual test costs against baselines.
plt.figure(figsize=(7, 4))
plt.scatter(test_idx, y_test, label="Actual test costs")
plt.plot(test_idx, mean_pred, "--", label="Mean baseline")
plt.plot(test_idx, median_pred, ":", label="Median baseline")

# Label the engineering context clearly.
plt.xlabel("Held-out bridge component index")
plt.ylabel("Repair cost, thousand dollars")
plt.title("Regression baselines for bridge repair cost")
plt.legend()

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **3.3. Regression Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_03_03.jpg?v=1783612908" width="250">



>* Simple average predictions set a baseline
>* Complex models should clearly outperform it

>* Baselines reveal real predictive improvement
>* Simple rules justify added model complexity

>* Compare improvements before adopting complex models
>* Use fair testing to support honest claims



In [ ]:
#@title Python Code - Regression Baselines

# Regression baselines compare simple prediction rules.
# Civil engineering targets are often continuous.
# Training data must define every baseline.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny concrete strength dataset.
cement_kg = np.array([260, 280, 300, 320, 340, 360, 380, 400])
strength_mpa = np.array([24, 27, 29, 31, 35, 36, 39, 42])

# Check matching feature and target lengths.
assert cement_kg.shape[0] == strength_mpa.shape[0]
assert strength_mpa.size >= 4

# Make a manual deterministic train test split.
train_idx = np.array([0, 1, 2, 3, 4, 5])
test_idx = np.array([6, 7])

# Separate training and testing target values.
y_train = strength_mpa[train_idx]
y_test = strength_mpa[test_idx]

# Build mean and median regression baselines.
mean_prediction = float(np.mean(y_train))
median_prediction = float(np.median(y_train))

# Predict the same baseline value for every test case.
y_mean_pred = np.full(y_test.shape, mean_prediction)
y_median_pred = np.full(y_test.shape, median_prediction)

# Define mean absolute error using NumPy.
mean_mae = np.mean(np.abs(y_test - y_mean_pred))
median_mae = np.mean(np.abs(y_test - y_median_pred))

# Print a compact baseline comparison.
print(f"Training mean baseline: {mean_prediction:.1f} MPa")
print(f"Training median baseline: {median_prediction:.1f} MPa")
print(f"Test MAE using mean baseline: {mean_mae:.1f} MPa")
print(f"Test MAE using median baseline: {median_mae:.1f} MPa")

# Plot actual test values and baseline predictions.
plt.figure(figsize=(6, 4))
plt.plot(cement_kg[test_idx], y_test, "o-", label="Actual test strength")
plt.plot(cement_kg[test_idx], y_mean_pred, "s--", label="Mean baseline")
plt.plot(cement_kg[test_idx], y_median_pred, "d--", label="Median baseline")
plt.xlabel("Cement content, kg per cubic meter")
plt.ylabel("Compressive strength, MPa")
plt.title("Regression baselines for concrete strength")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Workflow and Baselines**</font>


In this lecture, you learned to:
- Describe the steps in an end-to-end machine learning workflow for civil engineering data. 
- Implement a manual train/test split using simple Python and NumPy operations. 
- Build baseline predictors for regression and classification before training complex models. 

In the next Lecture (Lecture B), we will go over 'Metrics and Errors'